# Fixed CSA epoch 9 rerank on VALID5000

Attach visual-cache, mscoco-2014, csa-prompt-full-karpathy-v175, objectdetectionecache and resume Output with epoch 9 best.bin. T4 x2, Internet, Save & Run All. Fixed VALID164 weights 0.3 decoder / 0.4 CLIP / 0.3 object consistency. Scores top1 and rerank on VALID only. Send csa_per_caption_best9_rerank_valid5000_review.zip.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
REVISION='aae9afc7966ee4aa36dd67c4b4839ff788e775fe'
MAPPING={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'TEST199_LOCK.json', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': 'TEST199_PROTOCOL.md', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': 'TEST199_BUILD_V2_PROTOCOL.md', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': 'SMOKE_SOURCE.json', 'experiments/clip_patch_baseline/test199_data.py': 'test199_data.py', 'experiments/clip_patch_baseline/run_test199.py': 'run_test199.py', 'experiments/clip_patch_baseline/test_test199.py': 'test_test199.py', 'experiments/clip_patch_baseline/patch_model.py': 'patch_model.py', 'experiments/clip_patch_baseline/patch_beam.py': 'patch_beam.py', 'experiments/clip_patch_baseline/patch_data.py': 'patch_data.py', 'experiments/clip_patch_baseline/full_data.py': 'full_data.py', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'make_test199_notebook.py', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/cuda_runtime_guard.py': 'cuda_runtime_guard.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'experiments/csa_paper/score_prompt_test_extra.py': 'score_prompt_test_extra.py', 'experiments/clip_patch_baseline/semantic_test_data.py': 'semantic_test_data.py', 'experiments/clip_patch_baseline/semantic_data.py': 'semantic_data.py', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'semantic_patch_model.py', 'experiments/clip_patch_baseline/test_semantic_test.py': 'test_semantic_test.py', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': 'BANK_PROMPT_TEST_PROTOCOL.md', 'experiments/csa_paper/model.py': 'bank_layers.py', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'PROMPT_TEST_PREPARATION_RESULT.json'}
HASHES={'experiments/clip_patch_baseline/TEST199_LOCK.json': 'a93e320cc0c1444dab2de21ea2765107b949dc6f49353fd8c9c458ad8b60e8e3', 'experiments/clip_patch_baseline/TEST199_PROTOCOL.md': '2cd1f33ae85544688b4dca30e0e3a31da37454178044dfc48419030e45bde988', 'experiments/clip_patch_baseline/TEST199_BUILD_V2_PROTOCOL.md': '7213afcf56f4df32fdae98f72e35f1516d3d7cdcdf657b4b388a6e724af011d8', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': '6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49', 'experiments/clip_patch_baseline/test199_data.py': '0291e01b2f0f110e937904d40f4b5d1e69351605a2cff5c0cde55aef0fa412c4', 'experiments/clip_patch_baseline/run_test199.py': '3108367500cc3de3867d02ee5e0240b646a93f38aca184a79118964736d8e7f3', 'experiments/clip_patch_baseline/test_test199.py': 'ff7e3942a9e0db932694bff15a3485b2e132163731879871cbb1c5f611ac79be', 'experiments/clip_patch_baseline/patch_model.py': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70', 'experiments/clip_patch_baseline/patch_beam.py': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be', 'experiments/clip_patch_baseline/patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'experiments/clip_patch_baseline/full_data.py': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74', 'experiments/clip_patch_baseline/make_test199_notebook.py': 'd2d3a5a3d2bda3268732396f920f847b9fab5b11c60799197a0b667673481a64', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'experiments/csa_paper/score_prompt_test_extra.py': '8704765eb1c3c2713a7d31b92cba7454e905fcb3f0af37de627d81dc6d0f9bd7', 'experiments/clip_patch_baseline/semantic_test_data.py': 'a0f23b6d85a902a7f362a7a2ef03108fbaf603dc4326e61aeabbfe164bb3abdc', 'experiments/clip_patch_baseline/semantic_data.py': 'a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'e028734b1835ca41b51a63c838ebb05bde658a5e34ed0f6d1ea08796d1d24b8a', 'experiments/clip_patch_baseline/test_semantic_test.py': 'fd8756b0f2fa22efa35c5b79498fd5b1daffb5a0b18c25e8655b03f60927b951', 'experiments/clip_patch_baseline/BANK_PROMPT_TEST_PROTOCOL.md': '56e233a6d86dbfe269e21ced39c3253a0712cfe87cfb72399441728038857e7f', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/PROMPT_TEST_PREPARATION_RESULT.json': 'd99891923634188aa2d75185f0ef9894e3ab9e3be780a5a17e411076c3c9d205'}
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
PROMPT_TEST_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175')
CHECKPOINT_ROOT='' # Optional mounted best.bin or folder. Blank scans attached inputs.
WALL_HOURS=3 # STOP cap including CPU metrics, not ETA or billing.
OUTPUT=Path('/kaggle/working/csa_per_caption_best9_rerank_valid5000')
for p in (VISUAL_CACHE,COCO_JSON,PROMPT_TEST_ROOT):
    if not p.exists():raise FileNotFoundError('Attach/correct original inputs: '+str(p))
if OUTPUT.exists():raise RuntimeError('Preserve existing captions/output; use fresh session/output path')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<6*1024**3:raise RuntimeError('Need6GiB free for TEST patches/checkpoint/CPU scorer assets')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='clip-patch199-test-',dir='/tmp'));DEPS=SOURCE/'deps';CLI=SOURCE/'cli';CHAIR=SOURCE/'chair';LOG=OUTPUT/'notebook.log'
PARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600;TOKENS=[]
os.environ['PYTHONPATH']=str(SOURCE);os.environ['OMP_NUM_THREADS']='1'
CONFIG=SOURCE/'config.json'
cfg=dict(semantic=True,prompt_test_root=str(PROMPT_TEST_ROOT),output=str(OUTPUT),stage=str(SOURCE/'test_patches'),dataset_json=str(COCO_JSON),visual_cache=str(VISUAL_CACHE),checkpoint_root=CHECKPOINT_ROOT,chair=str(CHAIR),worker_seconds=1800)
def write_config():
    CONFIG.write_text(json.dumps(cfg,indent=2));(OUTPUT/'protocol.json').write_text(json.dumps(cfg,indent=2))
write_config()
def redact(line):
    for token in TOKENS:line=line.replace(token,'[REDACTED]')
    return line
def download(url,expected,path):
    remaining=PARENT_DEADLINE-time.monotonic()
    if remaining<=0:raise TimeoutError('Parent deadline before download')
    raw=urllib.request.urlopen(url,timeout=min(120,remaining)).read()
    if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Changed pinned input: '+path.name)
    path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(raw)

TRAIN_REVISION='5eb012279e6fd0b5e47a56710ed5e3efa7449d2f'
TRAIN_PINS={'experiments/clip_patch_baseline/csa_patch_model.py': '99b6441c4205b142256192cc3d840a905e00ecc1e730f30e1f5eb2c3cf8dc220', 'experiments/clip_patch_baseline/memory_fusion.py': 'a70947e22263ddd27a7896a94cfb35cfff73102370288c53f86529964ea8f087', 'experiments/clip_patch_baseline/semantic_patch_model.py': '10a385859ea1389a71f14a7a76e16a6989feb73fd581ca89659eab50ec53a0f3', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09'}

DETECTIONS=Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
if not DETECTIONS.is_file():raise FileNotFoundError('Attach objectdetectionecache or update DETECTIONS')
os.environ.update(TOKENIZERS_PARALLELISM='false',USE_TF='0',USE_FLAX='0',HF_HUB_DISABLE_XET='1')


In [ ]:
def run(args, timeout=1800, environment=None):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('TEST parent cap')
    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, env=environment, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: line=redact(line); print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('TEST/CPU metrics active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('TEST parent/phase cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
ROUTING_PREFLIGHT="import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi=KaggleApi()\nwith tempfile.TemporaryDirectory() as directory,patch.object(api,'build_kaggle_client') as factory:\n    endpoint=factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value=SimpleNamespace(files=[],next_page_token=None,log=None)\n    api.kernels_output('csa_structure_per_caption_full_xe10_resume124177',path=directory)\n    request=endpoint.call_args.args[0]\n    assert (request.user_name,request.kernel_slug,request.version_label)==('ducanh2403','kltnnnnnff','v9')\nprint('Exact own9 API routing mock PASS; no network.',flush=True)\n"
complete=False
try:
    for path,name in MAPPING.items():download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+path,HASHES[path],SOURCE/name)
    for path,digest in TRAIN_PINS.items():
        name='bank_layers.py' if path=='experiments/csa_paper/model.py' else Path(path).name
        download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+TRAIN_REVISION+'/'+path,digest,SOURCE/name)
    (SOURCE/'csa_per_caption_test_data.py').write_text('"""Fixed best9 TEST prompt/patch contracts; NumPy only, no model execution."""\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom real_bank_data import read,save,sha\nimport test199_data as shared\nfrom semantic_data import ARRAYS,record_arrays,checked_root\n\nHERE=Path(__file__).resolve().parent\nCHECKPOINT_SHA=\'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef\'\nVERSION=\'csa_structure_per_caption_full_xe10_resume124177\'\n\n\ndef validate_lock(value):\n    baseline=valid_cohort()\n    if any(value[k]!=baseline[k] for k in (\'expected_test_ids\',\'test_id_sha256\',\'beam_width\',\'max_new_tokens\',\'length_normalization\',\'reranking\')):raise ValueError(\'Changed fixed TEST cohort/decode\')\n    if value[\'source_version\']!=VERSION or value[\'checkpoint_sha256\']!=CHECKPOINT_SHA or value[\'epoch\']!=9 or value[\'updates\']!=159399 or value[\'method\']!=\'csa_structure_patch_prompt_grouped_per_caption_accelerate_v1\':raise ValueError(\'Need fixed bank+prompt best9, not endpoint10\')\n    return value\n\n\ndef valid_cohort():\n    import hashlib\n    value=shared.lock()\n    value.update(expected_test_ids=VALID_IDS,test_id_sha256=hashlib.sha256(json.dumps(VALID_IDS,separators=(\',\',\':\')).encode()).hexdigest())\n    return value\n\ndef lock():\n    value=valid_cohort();value.update(schema=\'csa_per_caption_best9_test_v1\',source_version=VERSION,checkpoint_sha256=CHECKPOINT_SHA,\n        training_commit=\'5eb012279e6fd0b5e47a56710ed5e3efa7449d2f\',epoch=9,updates=159399,method=\'csa_structure_patch_prompt_grouped_per_caption_accelerate_v1\')\n    return validate_lock(value)\n\n\ndef checkpoint(root):\n    root=Path(root);files=[root] if root.is_file() else sorted(root.rglob(\'best.bin\'))\n    matches=[p for p in files if p.is_file() and sha(p)==CHECKPOINT_SHA]\n    if not matches:raise ValueError(\'Missing exact version9 best.bin SHAb0c5bde0; no final/latest fallback\')\n    return matches[0]\n\n\ndef runtime_policy(training,current):\n    if training!=\'2.11.0+cu128\' or current not in (\'2.11.0+cu128\',\'2.11.0+cu130\'):raise RuntimeError(\'Unreviewed Torch build; keep log, do not blindly reinstall.\')\n    return dict(training_torch=training,inference_torch=current,cuda_wheel_changed=training!=current,\n        policy=\'Same publicTorch2.11.0 observed CUDA wheels; strictstate and actual GPU beam-path check required.\',bitwise_cross_build_equivalence_established=False)\n\n\ndef validate_checkpoint_state(ck):\n    s,i=ck[\'state\'],ck[\'identity\'];expected=lock()\n    if s[\'epoch\']!=10 or s[\'batch_cursor\']!=0 or s[\'phase\']!=\'train\' or s[\'updates\']!=159399 or s[\'caption_events\']!=5100723 or s[\'best_epoch\']!=9 or len(s[\'history\'])!=9:raise ValueError(\'Wrong best9 snapshot cursor\')\n    if i[\'source\'][\'commit\']!=expected[\'training_commit\'] or i[\'method\']!=expected[\'method\'] or i[\'epochs\']!=10 or i[\'global_batch\']!=32:raise ValueError(\'Wrong checkpoint architecture/training identity\')\n    if not {\'core.nouns\',\'core.verbs\'}<=set(ck[\'model\']):raise ValueError(\'Missing frozen TRAIN banks in checkpoint buffers\')\n    return expected\n\n\ndef configure_valid(cfg):\n    global VALID_IDS\n    if sha(cfg[\'dataset_json\'])!=shared.DATA_SHA:raise ValueError(\'Changed Karpathy metadata\')\n    def val_rows(ds, locked):\n        rows=sorted((r for r in ds[\'images\'] if r[\'split\']==\'val\'),key=lambda r:int(r[\'cocoid\']))\n        if [int(r[\'cocoid\']) for r in rows]!=locked[\'expected_test_ids\']:raise ValueError(\'Wrong VALID IDs\')\n        return rows\n    VALID_IDS=sorted(int(r[\'cocoid\']) for r in read(cfg[\'dataset_json\'])[\'images\'] if r[\'split\']==\'val\')\n    if len(VALID_IDS)!=5000 or len(set(VALID_IDS))!=5000:raise ValueError(\'Need VALID5000\')\n    shared.selected_test_rows=val_rows\n\ndef prompt_inputs(cfg):\n    configure_valid(cfg)\n    inv=read(HERE/\'FULL_PROMPT_FRESH_V175_RESULT.json\')[\'transport_inventory\'][\'cache_files\']\n    wanted={n:inv[n] for n in (\'cache_receipt.json\',\'cohort.json\',\'prompt_links.jsonl\',\'prompt_texts.json\',\'prompt_embeddings.npy\')}\n    root=checked_root(cfg[\'prompt_test_root\'],\'cache_receipt.json\',wanted)\n    cohort=read(root/\'cohort.json\')\n    if cohort[\'test\'] or sorted(cohort[\'val\'])!=VALID_IDS:raise ValueError(\'Wrong prompt175 VALID cohort\')\n    texts=read(root/\'prompt_texts.json\')\n    vectors=np.load(root/\'prompt_embeddings.npy\',allow_pickle=False)\n    if texts!=sorted(set(texts)) or vectors.shape!=(410,512) or vectors.dtype!=np.float32 or not np.isfinite(vectors).all():raise ValueError(\'Changed prompt175 embeddings\')\n    with (root/\'prompt_links.jsonl\').open(encoding=\'utf8\') as stream:\n        rows=[r for r in map(json.loads,stream) if r[\'group\']==\'val\']\n    if sorted(r[\'coco_id\'] for r in rows)!=VALID_IDS:raise ValueError(\'Wrong VALID prompt links\')\n    return rows,texts,vectors,wanted\n\ndef prepare(cfg):\n    rows,texts,vectors,seals=prompt_inputs(cfg)\n    for row in rows:record_arrays(row,texts)\n    shared.prepare(cfg,locked=lock(),weight_path=checkpoint(cfg[\'checkpoint_root\']))\n    stage_prompts(cfg,rows,texts,vectors,seals)\n\n\ndef stage_prompts(cfg,rows,texts,vectors,seals):\n    encoded=[record_arrays(row,texts) for row in rows]\n    stage=Path(cfg[\'stage\']);out=Path(cfg[\'output\']);ids=np.load(stage/\'ids.npy\',allow_pickle=False);lookup={r[\'coco_id\']:j for j,r in enumerate(rows)}\n    order=[lookup[int(i)] for i in ids]\n    for key in ARRAYS:np.save(stage/(\'sem_\'+key+\'.npy\'),np.stack([encoded[j][key] for j in order]))\n    np.save(stage/\'sem_text_vectors.npy\',vectors)\n    receipt=read(out/\'stage_receipt.json\');receipt[\'method\']=\'csa_structure_patch_prompt_grouped_per_caption_accelerate_v1\';receipt[\'prompt175_files_sha256\']=seals\n    receipt[\'stage_files_sha256\'].update({p.name:sha(p) for p in stage.glob(\'sem_*.npy\')});save(out/\'stage_receipt.json\',receipt)\n    saved=read(out/\'test_lock.json\');saved[\'stage_receipt_sha256\']=sha(out/\'stage_receipt.json\');save(out/\'test_lock.json\',saved)\n    save(out/\'semantic_test_stage.json\',dict(images=5000,objects=sum(len(r[\'objects\']) for r in rows),relations=sum(len(r[\'relations\']) for r in rows),\n        text_vectors=410,geometry=\'same training record_arrays ROI box->fractional14x14patch support; endpoint IDs\',TRAIN_TEST_features_read=False,old_ROI_features_used=False,old_vocabulary_used=False))\n\n\ndef load(cfg,verify=True):\n    result=shared.load(cfg,verify=verify,validator=validate_lock)\n    if result[2].get(\'method\')!=\'csa_structure_patch_prompt_grouped_per_caption_accelerate_v1\':raise ValueError(\'Semantic TEST stage incomplete\')\n    return result\n',encoding='utf8')
    (SOURCE/'run_csa_per_caption_test.py').write_text('"""USER-RUN fixed199 inference and CPU scoring. No training mode."""\nimport argparse\nfrom datetime import timedelta\nimport importlib.metadata\nimport math\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom test199_data import prepare,load,lock,validate_lock,checkpoint,runtime_policy,CHECKPOINT_SHA,TEST_SIZE\nfrom saved_quality_data import check_rows,chair_counts\n\ndef contracts(cfg):\n    if cfg.get(\'semantic\'):\n        import csa_per_caption_test_data as module\n        module.configure_valid(cfg)\n    else:\n        import test199_data as module\n    return module\n\n\ndef checked_weights(cfg):\n    import torch\n    d=contracts(cfg);ck=torch.load(d.checkpoint(cfg[\'checkpoint_root\']),map_location=\'cpu\',weights_only=False)\n    if cfg.get(\'semantic\'):\n        d.validate_checkpoint_state(ck)\n        return ck,d.runtime_policy(ck[\'identity\'][\'software\'][\'torch\'],str(torch.__version__))\n    if ck[\'state\'][\'updates\']!=177110 or ck[\'state\'][\'best_epoch\']!=10 or ck[\'identity\'][\'source\'][\'commit\']!=lock()[\'training_commit\']:raise ValueError(\'Unexpected fixed199 training state\')\n    return ck,runtime_policy(ck[\'identity\'][\'software\'][\'torch\'],str(torch.__version__))\n\ndef checkpoint_preflight(cfg):\n    ck,policy=checked_weights(cfg);del ck\n    save(Path(cfg[\'output\'])/\'checkpoint_runtime_preflight.json\',dict(checkpoint_sha256=contracts(cfg).CHECKPOINT_SHA,runtime_policy=policy,models_executed=False))\n    print(\'Checkpoint/runtime metadata PASS:\',policy,flush=True)\n\ndef worker(cfg):\n    import torch\n    import torch.distributed as dist\n    from torch.nn import functional as F\n    from patch_model import PatchCaptioner\n    from patch_beam import beam_search,check_paths,text_from_ids\n    from patch_data import batch as caption_batch\n    from cuda_runtime_guard import check_runtime\n    rank=int(os.environ[\'LOCAL_RANK\']);out=Path(cfg[\'output\']);check_runtime(out,rank)\n    if int(os.environ[\'WORLD_SIZE\'])!=2:raise ValueError(\'Use T4x2\')\n    torch.set_num_threads(1);torch.cuda.set_device(rank);device=torch.device(\'cuda\',rank)\n    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False\n    dist.init_process_group(\'nccl\',timeout=timedelta(seconds=1200));started=time.monotonic();deadline=started+cfg[\'worker_seconds\']\n    try:\n        d=contracts(cfg);data,words,receipt,locked=d.load(cfg,verify=rank==0);dist.barrier(device_ids=[rank])\n        ck,policy=checked_weights(cfg);save(out/f\'runtime_policy_rank{rank}.json\',policy)\n        context=None\n        if cfg.get(\'semantic\'):\n            from csa_patch_model import CSAPatchCaptioner\n            from semantic_data import SemanticContext\n            model=CSAPatchCaptioner(len(words),ck[\'model\'][\'core.nouns\'],ck[\'model\'][\'core.verbs\']);context=SemanticContext(cfg[\'stage\'])\n        else:model=PatchCaptioner(len(words))\n        model=model.to(device);model.load_state_dict(ck[\'model\'],strict=True);del ck;model.eval();rows=[];beams=[];images=np.arange(TEST_SIZE)[rank::2]\n        with torch.inference_mode():\n            for offset in range(0,len(images),8):\n                if time.monotonic()>deadline-30:raise TimeoutError(\'TEST decode phase deadline\')\n                index=images[offset:offset+8];x=torch.from_numpy(np.asarray(data[\'patches\'][index],np.float32)).to(device)\n                memory=model.encode(x,context.images(index,device)) if context else model.encode(x);seq,scores=beam_search(model,memory,width=locked[\'beam_width\'],maximum=locked[\'max_new_tokens\'])\n                if offset==0:check_paths(model,memory,seq,scores)\n                for i,seqs,values in zip(index.tolist(),seq.cpu().tolist(),scores.cpu().tolist()):\n                    beams.append(dict(coco_id=int(data[\'ids\'][i]),candidates=[\n                        dict(ids=ids,caption=text_from_ids(ids,words),logprob=float(value))\n                        for ids,value in zip(seqs,values)]))\n                for i,s,v in zip(index.tolist(),seq[:,0].cpu().tolist(),scores[:,0].cpu().tolist()):rows.append(dict(coco_id=int(data[\'ids\'][i]),caption=text_from_ids(s,words),ids=s,score=float(v)))\n                if offset%200==0:print(\'TEST decode rank\',rank,len(rows),\'/\',len(images),flush=True)\n            save(out/f\'captions_rank{rank}.json\',rows)\n            save(out/f\'beams_rank{rank}.json\',beams)\n            nll_sum=0.;tokens=0;ev=np.flatnonzero(np.isin(data[\'caption_image\'],images))\n            for offset in range(0,len(ev),32):\n                if time.monotonic()>deadline-30:raise TimeoutError(\'TEST teacherNLL phase deadline; captions preserved\')\n                x,seq=caption_batch(data,ev[offset:offset+32]);x=torch.from_numpy(x).to(device);seq=torch.from_numpy(seq).to(device)\n                logits=model(x,seq[:,:-1],context.events(data,ev[offset:offset+32],device)) if context else model(x,seq[:,:-1]);nll_sum+=float(F.cross_entropy(logits.flatten(0,1),seq[:,1:].flatten(),ignore_index=0,reduction=\'sum\'));tokens+=int((seq[:,1:]!=0).sum())\n        if len(rows)!=2500:raise ValueError(\'Incomplete TEST rank\')\n        save(out/f\'worker{rank}.json\',dict(complete=True,rank=rank,images=len(rows),NLL_sum=nll_sum,NLL_tokens=tokens,seconds=time.monotonic()-started,training=False,checkpoint_sha256=d.CHECKPOINT_SHA,caption_sha256=sha(out/f\'captions_rank{rank}.json\'),beams_sha256=sha(out/f\'beams_rank{rank}.json\'),runtime_policy=policy,first_GPU_beam_path_check_passed=True))\n        dist.barrier(device_ids=[rank])\n    finally:\n        if dist.is_initialized():dist.destroy_process_group()\n\ndef inputs(cfg):\n    d=contracts(cfg);out=Path(cfg[\'output\']);locked=d.lock();saved=read(out/\'test_lock.json\');d.validate_lock(saved);receipt=read(out/\'stage_receipt.json\')\n    if saved[\'stage_receipt_sha256\']!=sha(out/\'stage_receipt.json\') or sha(out/\'test_references.json\')!=receipt[\'output_files_sha256\'][\'test_references.json\']:raise ValueError(\'Changed TEST references/receipt\')\n    refs=read(out/\'test_references.json\');workers=[read(out/f\'worker{r}.json\') for r in (0,1)]\n    if not all(w[\'complete\'] and w[\'images\']==2500 and w[\'checkpoint_sha256\']==d.CHECKPOINT_SHA for w in workers):raise ValueError(\'Incomplete TEST model output\')\n    if workers[0][\'runtime_policy\']!=workers[1][\'runtime_policy\'] or not all(w[\'first_GPU_beam_path_check_passed\'] for w in workers):raise ValueError(\'Runtime/beam guards differ between ranks\')\n    rows=[]\n    for rank,w in enumerate(workers):\n        if sha(out/f\'captions_rank{rank}.json\')!=w[\'caption_sha256\']:raise ValueError(\'Changed TEST rank captions\')\n        rows+=read(out/f\'captions_rank{rank}.json\')\n    check_rows(rows,locked[\'expected_test_ids\'])\n    if len(refs)!=5000 or {r[\'coco_id\'] for r in refs}!=set(locked[\'expected_test_ids\']):raise ValueError(\'Wrong TEST references\')\n    return out,refs,rows,workers\n\ndef tokenize(cfg):\n    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n    out,refs,rows,workers=inputs(cfg);rows=sorted(rows,key=lambda r:r[\'coco_id\']);save(out/\'captions.json\',rows)\n    tok=PTBTokenizer();gts=tok.tokenize({r[\'coco_id\']:[dict(caption=t) for t in r[\'references\']] for r in refs});hyp=tok.tokenize({r[\'coco_id\']:[dict(caption=r[\'caption\'])] for r in rows})\n    save(out/\'reference_ptb.json\',gts);save(out/\'caption_ptb.json\',hyp)\n    save(out/\'scoring_lock.json\',dict(checkpoint_sha256=contracts(cfg).CHECKPOINT_SHA,files_sha256={n:sha(out/n) for n in (\'captions.json\',\'test_references.json\',\'test_lock.json\',\'reference_ptb.json\',\'caption_ptb.json\')},training=False,reranking=False))\n\ndef score(cfg,mode):\n    out=Path(cfg[\'output\']);sealed=read(out/\'scoring_lock.json\')\n    for n,h in sealed[\'files_sha256\'].items():\n        if sha(out/n)!=h:raise ValueError(\'Changed scoring input \'+n)\n    gts=read(out/\'reference_ptb.json\');hyp=read(out/\'caption_ptb.json\')\n    if len(gts)!=5000 or set(gts)!=set(hyp):raise ValueError(\'Incomplete TEST scorer cohort\')\n    result={}\n    if mode==\'basic\':\n        from pycocoevalcap.bleu.bleu import Bleu\n        from pycocoevalcap.cider.cider import Cider\n        from pycocoevalcap.rouge.rouge import Rouge\n        b,_=Bleu(4).compute_score(gts,hyp);c,per=Cider().compute_score(gts,hyp);r,_=Rouge().compute_score(gts,hyp)\n        result=dict(BLEU_1=float(b[0]),BLEU_4=float(b[3]),CIDEr=float(c),ROUGE_L=float(r));save(out/\'cider_per_image.json\',[dict(coco_id=int(i),CIDEr=float(v)) for i,v in zip(gts,per)])\n    elif mode in (\'meteor\',\'spice\'):\n        if mode==\'meteor\':\n            from pycocoevalcap.meteor.meteor import Meteor\n            scorer=Meteor()\n        else:\n            from pycocoevalcap.spice.spice import Spice\n            scorer=Spice()\n        try:point,per=scorer.compute_score(gts,hyp)\n        finally:del scorer\n        if len(per)!=5000:raise ValueError(\'Incomplete \'+mode)\n        result={mode.upper():float(point)}\n        values=[float(x) for x in per] if mode==\'meteor\' else [float(x[\'All\'][\'f\']) for x in per]\n        if not all(math.isfinite(x) for x in values):raise ValueError(\'Nonfinite per-image \'+mode)\n        save(out/(mode+\'_per_image.json\'),[dict(coco_id=int(i),score=v) for i,v in zip(gts,values)])\n    elif mode==\'chair\':\n        rows=read(out/\'captions.json\');save(out/\'chair_input.json\',[dict(image_id=r[\'coco_id\'],caption=r[\'caption\']) for r in rows]);chair=Path(cfg[\'chair\'])\n        subprocess.run([sys.executable,\'-u\',chair/\'chair.py\',\'--cap_file\',out/\'chair_input.json\',\'--cache\',chair/\'chair.pkl\',\'--save_path\',out/\'chair_output.json\'],cwd=chair,check=True,timeout=600)\n        result,counts=chair_counts(read(out/\'chair_output.json\'),rows);save(out/\'chair_counts.json\',counts)\n    else:raise ValueError(mode)\n    if not all(math.isfinite(v) for v in result.values()):raise ValueError(\'Nonfinite metric\')\n    save(out/(mode+\'.json\'),dict(images=5000,metrics=result,input_sha256=sealed[\'files_sha256\']))\n\ndef combine(cfg):\n    out,refs,rows,workers=inputs(cfg);sealed=read(out/\'scoring_lock.json\');metrics={}\n    for n,h in sealed[\'files_sha256\'].items():\n        if sha(out/n)!=h:raise ValueError(\'Changed final scoring input \'+n)\n    for mode in (\'basic\',\'meteor\',\'spice\',\'chair\'):\n        p=read(out/(mode+\'.json\'))\n        if p[\'images\']!=5000 or p[\'input_sha256\']!=sealed[\'files_sha256\']:raise ValueError(\'Incomplete/changed metric phase\')\n        metrics.update(p[\'metrics\'])\n    required={\'BLEU_1\',\'BLEU_4\',\'METEOR\',\'ROUGE_L\',\'CIDEr\',\'SPICE\',\'CHAIRs\',\'CHAIRi\',\'Recall\'}\n    if set(metrics)!=required or not all(math.isfinite(v) for v in metrics.values()):raise ValueError(\'Need all eight metrics and recall\')\n    metrics[\'NLL\']=sum(w[\'NLL_sum\'] for w in workers)/sum(w[\'NLL_tokens\'] for w in workers)\n    save(out/\'result.json\',dict(status=\'CSA_PER_CAPTION_BEST9_TEST5000_COMPLETE\' if cfg.get(\'semantic\') else \'CLIP_PATCH_199_TEST5000_COMPLETE\',images=5000,checkpoint_sha256=contracts(cfg).CHECKPOINT_SHA,source_version=contracts(cfg).lock()[\'source_version\'],epoch=contracts(cfg).lock()[\'epoch\'],metrics=metrics,display_x100={k:100*v for k,v in metrics.items() if k!=\'NLL\'},beam_width=5,max_new_tokens=20,training=False,reranking=False,test_tuning=False,prior_project_test_exposure=True,chair_counts=read(out/\'chair_counts.json\'),environment=read(out/\'environment.json\'),runtime_policy=workers[0][\'runtime_policy\']))\n    print(\'TEST5000 COMPLETE\',metrics,flush=True)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'mode\',choices=(\'checkpoint_preflight\',\'prepare\',\'worker\',\'environment\',\'smoke\',\'tokenize\',\'basic\',\'meteor\',\'spice\',\'chair\',\'combine\'));p.add_argument(\'--config\',required=True);a=p.parse_args();cfg=read(a.config)\n    if a.mode in (\'environment\',\'smoke\',\'basic\',\'meteor\',\'spice\',\'chair\'):\n        import score_prompt_test_extra as cpu\n        if a.mode==\'environment\':cpu.environment(Path(cfg[\'output\']))\n        else:\n            cpu.java_path(Path(cfg[\'output\']))\n            if a.mode==\'smoke\':cpu.smoke(Path(cfg[\'output\']))\n            else:score(cfg,a.mode)\n    elif a.mode==\'tokenize\':\n        import score_prompt_test_extra as cpu\n        cpu.java_path(Path(cfg[\'output\']));tokenize(cfg)\n    elif a.mode==\'prepare\':contracts(cfg).prepare(cfg)\n    else:globals()[a.mode](cfg)\n',encoding='utf8')
    (OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES,training_commit=TRAIN_REVISION,training_pins=TRAIN_PINS),indent=2))
    run([sys.executable,'-u',SOURCE/'test_test199.py'],120)
    run([sys.executable,'-u',SOURCE/'test_semantic_test.py'],120)
    run([sys.executable,'-u',SOURCE/'cuda_runtime_guard.py','--output',OUTPUT],60)
    sys.path.insert(0,str(SOURCE))
    from csa_per_caption_test_data import checkpoint,VERSION,CHECKPOINT_SHA,prompt_inputs
    weights=checkpoint(CHECKPOINT_ROOT or '/kaggle/input')
    origin='mounted resume Output, verified by SHA256'
    cfg['checkpoint_root']=str(weights);write_config()
    download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/5eb012279e6fd0b5e47a56710ed5e3efa7449d2f/experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json','452e94530b07b117be6873f004404b558e29e87cb09e97e27a1548d0bfd5f81a',SOURCE/'FULL_PROMPT_FRESH_V175_RESULT.json')
    prompt_inputs(cfg)
    (OUTPUT/'checkpoint_source.json').write_text(json.dumps(dict(version=VERSION,sha256=CHECKPOINT_SHA,path=str(weights),origin=origin),indent=2))
    run([sys.executable,'-u',SOURCE/'run_csa_per_caption_test.py','checkpoint_preflight','--config',CONFIG],120)
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,'pycocoevalcap==1.2','nltk==3.9.2','future'],600)
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,'git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'],600)
    os.environ['PYTHONPATH']=str(DEPS)+os.pathsep+str(SOURCE)
    run([sys.executable,'-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"],300)
    base='https://raw.githubusercontent.com/Maxlinn/CHAIR-metric-standalone/4087a26211aa2339b9a76307cb8f0321ef691d0a/'
    for name,digest in {'chair.py':'98728cd2b7fb8b97420b5f3bf14789dcad1d7d980de9c089b82856217387770d','chair.pkl':'73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3'}.items():download(base+name,digest,CHAIR/name)
    run([sys.executable,'-u',SOURCE/'run_csa_per_caption_test.py','environment','--config',CONFIG],1200)
    run([sys.executable,'-u',SOURCE/'run_csa_per_caption_test.py','smoke','--config',CONFIG],600)
    run([sys.executable,'-u',SOURCE/'run_csa_per_caption_test.py','prepare','--config',CONFIG],1800)
    cfg['worker_seconds']=min(1800,PARENT_DEADLINE-time.monotonic()-1800);write_config()
    if cfg['worker_seconds']<300:raise TimeoutError('Insufficient budget before TEST inference; no automatic extension')
    launch=[sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0']
    run([*launch,SOURCE/'run_csa_per_caption_test.py','worker','--config',CONFIG],cfg['worker_seconds']+30)
    for path,digest in {'captioning/object_consistency.py': '5fbd75b0ff4a35f0475d460a294894009d6007f468e248a0e4b528aaa8fabf73', 'captioning/multiscore_reranking.py': 'd1baa60e244c86b32e721486d7923ead1391c3a2fb744651c257995f2d0bb443', 'experiments/csa_paper/RERANK_TEST_INPUTS.json': '33395ec35a0803ebf711737f0bdba3b6be8c16244f4c292057658e0fa3feaf46'}.items():
        target=SOURCE/path if path.startswith('captioning/') else SOURCE/Path(path).name
        download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/277851d0fbc4643ae7001bc7702dd0ce5c91d0f6/'+path,digest,target)
    (SOURCE/'REPLAY_RECEIPTS.json').write_text(json.dumps({'checkpoint_sha256': 'b0c5bde0921a4c828ac094b264f502654224aa9365ee40cdce1c15c82fe9aeef', 'rank_caption_sha256': ['e2eb3ad72d4b3ace9cd08ba82621936399a2700e0d63b1c8e1810e56eebcb6ee', '43c3b65a2c68aeb77700fe05cb51250ca930080df046f998e528deffc09a203b']}))
    (SOURCE/'csa_per_caption_rerank_data.py').write_text('"""Prepare and check inputs for the frozen CSA-only rerank replay."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nfrom captioning.object_consistency import index_detections\nfrom patch_beam import text_from_ids\nfrom real_bank_data import read, save, sha\nfrom test199_data import DATA_SHA\n\nHERE = Path(__file__).resolve().parent\nPOLICIES = (\'decoder_rank0\', \'clip_object_fixed\')\nDETECTION_SHA = \'498825daf8f8773a1c92f4111dc4ccf1f4296343f952b486e0aa56caf82ec04c\'\n\n\ndef prepare(cfg):\n    out = Path(cfg[\'output\'])\n    test = Path(cfg[\'test_output\'])\n    source = read(test/\'test_lock.json\')\n    policy = read(HERE/\'RERANK_TEST_INPUTS.json\')\n    replay = read(HERE/\'REPLAY_RECEIPTS.json\')\n    if source[\'checkpoint_sha256\'] != replay[\'checkpoint_sha256\']:\n        raise ValueError(\'Changed CSA checkpoint\')\n    if sha(cfg[\'dataset_json\']) != DATA_SHA or sha(cfg[\'detections\']) != DETECTION_SHA:\n        raise ValueError(\'Changed COCO metadata or detector\')\n    records = sorted((r for r in read(cfg[\'dataset_json\'])[\'images\'] if r[\'split\'] == \'val\'),\n                     key=lambda r: int(r[\'cocoid\']))\n    if [int(r[\'cocoid\']) for r in records] != source[\'expected_test_ids\']:\n        raise ValueError(\'Wrong TEST image cohort\')\n    detections = index_detections(read(cfg[\'detections\']))\n    root = Path(cfg[\'dataset_json\']).parent.resolve()\n    names = {r[\'filename\'] for r in records}\n    paths = {}\n    for p in root.rglob(\'*.jpg\'):\n        if p.name in names:\n            if p.name in paths or not p.resolve().is_relative_to(root):\n                raise ValueError(\'Ambiguous TEST JPEG: \'+p.name)\n            paths[p.name] = p\n    if set(paths) != names or not names <= detections.keys():\n        raise ValueError(\'Missing TEST JPEGs or detector rows\')\n    inputs = []\n    for row in records:\n        name = row[\'filename\']\n        path = paths[name]\n        inputs.append(dict(coco_id=int(row[\'cocoid\']), image_path=str(path),\n                           image_sha256=sha(path), detected=detections[name]))\n    save(out/\'inference_inputs.json\', inputs)\n    (out/\'test_references.json\').write_bytes((test/\'test_references.json\').read_bytes())\n    save(out/\'test_lock.json\', dict(\n        checkpoint_sha256=replay[\'checkpoint_sha256\'], images=5000,\n        clip=policy[\'clip\'], selector=policy[\'frozen_selector\'],\n        inference_sha256=sha(out/\'inference_inputs.json\'),\n        references_sha256=sha(out/\'test_references.json\')))\n\n\ndef checked_lock(out):\n    out = Path(out)\n    lock = read(out/\'test_lock.json\')\n    policy = read(HERE/\'RERANK_TEST_INPUTS.json\')\n    replay = read(HERE/\'REPLAY_RECEIPTS.json\')\n    if (lock[\'clip\'], lock[\'selector\'], lock[\'checkpoint_sha256\'], lock[\'images\']) != (\n            policy[\'clip\'], policy[\'frozen_selector\'], replay[\'checkpoint_sha256\'], 5000):\n        raise ValueError(\'Changed rerank policy or cohort\')\n    if sha(out/\'inference_inputs.json\') != lock[\'inference_sha256\']:\n        raise ValueError(\'Changed inference inputs\')\n    return lock\n\n\ndef read_beams(out, lock):\n    out = Path(out)\n    test = out.parent\n    inputs = read(out/\'inference_inputs.json\')\n    words = read(test/\'vocabulary.json\')\n    by_id = {}\n    for rank in (0, 1):\n        path = test/f\'beams_rank{rank}.json\'\n        receipt = read(test/f\'worker{rank}.json\')\n        if (not receipt[\'complete\'] or receipt[\'images\'] != 2500\n                or receipt[\'checkpoint_sha256\'] != lock[\'checkpoint_sha256\']\n                or sha(path) != receipt[\'beams_sha256\']):\n            raise ValueError(\'Incomplete CSA beam replay\')\n        rows = read(path)\n        original = read(test/f\'captions_rank{rank}.json\')\n        if len(original) != 2500 or [r[\'coco_id\'] for r in rows] != [r[\'coco_id\'] for r in inputs[rank::2]]:\n            raise ValueError(\'Wrong TEST beam coverage\')\n        for row, top in zip(rows, original):\n            beams = row[\'candidates\']\n            if (len(beams) != 5 or row[\'coco_id\'] != top[\'coco_id\']\n                    or beams[0] != dict(ids=top[\'ids\'], caption=top[\'caption\'], logprob=top[\'score\'])):\n                raise ValueError(\'Top-1 does not match beam candidates\')\n            if any(c[\'caption\'] != text_from_ids(c[\'ids\'], words) for c in beams):\n                raise ValueError(\'Changed beam text\')\n            by_id[row[\'coco_id\']] = row\n    return [by_id[row[\'coco_id\']] for row in inputs]\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'config\')\n    prepare(read(parser.parse_args().config))\n',encoding='utf8')
    (SOURCE/'run_csa_per_caption_rerank.py').write_text("import argparse, hashlib, io, json, os, time\nfrom pathlib import Path\nimport numpy as np\nfrom real_bank_data import read,save,sha\nfrom csa_per_caption_rerank_data import checked_lock, read_beams, POLICIES\nfrom captioning.multiscore_reranking import rerank_multiscore\nfrom score_prompt_test_extra import java_path\n\ndef fixed_rank(row, clipscore, detected):\n    if len(row['candidates']) != 5 or len(clipscore) != 5 or any((not np.isfinite(x) or not 0 <= x <= 2.50001 for x in clipscore)):\n        raise ValueError('Invalid TEST CLIP scores/width')\n    scores = [c['logprob'] for c in row['candidates']]\n    if not all(np.isfinite(scores)) or scores != sorted(scores, reverse=True):\n        raise ValueError('Invalid beam logprobs')\n    enriched = [dict(c, clipscore=float(s)) for c, s in zip(row['candidates'], clipscore)]\n    chosen, _ = rerank_multiscore(dict(candidates=enriched), detected, 0.4, 0.3)\n    return int(chosen['original_rank'])\n\ndef prefetch(cfg):\n    from huggingface_hub import snapshot_download\n    out = Path(cfg['output'])\n    locked = checked_lock(out)['clip']\n    root = Path(snapshot_download(locked['id'], revision=locked['revision'], allow_patterns=list(locked['files_sha256']), cache_dir=cfg['model_cache']))\n    for n, h in locked['files_sha256'].items():\n        if sha(root / n) != h:\n            raise ValueError('CLIP assets differ from VALID164: ' + n)\n    save(out / 'clip_assets.json', dict(**locked, path=str(root)))\n    print('CLIP model/processor match frozen VALID164 bytes; no BLIP download.', flush=True)\n\ndef text_preflight(cfg):\n    from transformers import CLIPProcessor\n    out = Path(cfg['output'])\n    lock = checked_lock(out)\n    rows = read_beams(out, lock)\n    model = read(out / 'clip_assets.json')\n    processor = CLIPProcessor.from_pretrained(model['path'], local_files_only=True, use_fast=False)\n    lengths = [len(v) for v in processor.tokenizer([c['caption'] for r in rows for c in r['candidates']], truncation=False)['input_ids']]\n    if len(lengths) != 25000 or max(lengths) > 77:\n        raise ValueError('Would truncate TEST candidates')\n    save(out / 'token_coverage.json', dict(images=5000, captions=25000, max_tokens=max(lengths), limit=77))\n\ndef worker(cfg):\n    import torch\n    import torch.nn.functional as F\n    from PIL import Image\n    from transformers import CLIPModel, CLIPProcessor\n    rank = int(os.environ['LOCAL_RANK'])\n    if int(os.environ['WORLD_SIZE']) != 2 or torch.cuda.device_count() != 2:\n        raise ValueError('Select T4x2')\n    torch.cuda.set_device(rank)\n    device = torch.device('cuda', rank)\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    out = Path(cfg['output'])\n    lock = checked_lock(out)\n    asset = read(out / 'clip_assets.json')\n    for n, h in lock['clip']['files_sha256'].items():\n        if sha(Path(asset['path']) / n) != h:\n            raise ValueError('Changed CLIP model/processor')\n    base = CLIPModel.from_pretrained(asset['path'], local_files_only=True, use_safetensors=False, attn_implementation='eager').float().to(device).eval().requires_grad_(False)\n    processor = CLIPProcessor.from_pretrained(asset['path'], local_files_only=True, use_fast=False)\n    if sha(out / 'inference_inputs.json') != lock['inference_sha256']:\n        raise ValueError('Changed TEST inference inputs')\n    inputs = read(out / 'inference_inputs.json')\n    beams = read_beams(out, lock)\n    started = time.monotonic()\n    count = 0\n    path = out / f'scores_rank{rank}.jsonl'\n    with path.open('x', encoding='utf-8') as log, torch.inference_mode():\n        for row, beam in list(zip(inputs, beams))[rank::2]:\n            if time.monotonic() - started > cfg['phase_seconds']:\n                raise TimeoutError('TEST CLIP scoring cap; preserve partial scores')\n            raw = Path(row['image_path']).read_bytes()\n            if hashlib.sha256(raw).hexdigest() != row['image_sha256']:\n                raise ValueError('Changed raw TEST image')\n            with Image.open(io.BytesIO(raw)) as image:\n                image = image.convert('RGB')\n            pixels = processor(images=image, return_tensors='pt')['pixel_values'].to(device)\n            text = processor.tokenizer([c['caption'] for c in beam['candidates']], padding=True, truncation=False, return_tensors='pt').to(device)\n            image_features = F.normalize(base.get_image_features(pixel_values=pixels).float(), dim=-1)\n            text_features = F.normalize(base.get_text_features(**text).float(), dim=-1)\n            clipscore = (2.5 * (image_features * text_features).sum(-1).clamp_min(0)).cpu().tolist()\n            chosen = fixed_rank(beam, clipscore, row['detected'])\n            log.write(json.dumps(dict(coco_id=row['coco_id'], image_sha256=row['image_sha256'], clipscore=clipscore, selected_rank=chosen), allow_nan=False) + '\\n')\n            log.flush()\n            count += 1\n            if count % 100 == 0:\n                print('TEST CLIP GPU', rank, count, '/2500', flush=True)\n            if count == 25 and (time.monotonic() - started) / count * (2500 - count) > cfg['phase_seconds'] - (time.monotonic() - started):\n                raise TimeoutError('Measured TEST CLIP projection exceeds cap')\n    save(out / f'worker{rank}.json', dict(complete=True, images=count, scored_pairs=count * 5, scores_sha256=sha(path), seconds=time.monotonic() - started, torch_version=torch.__version__, training=False, clip_revision=lock['clip']['revision']))\n\ndef score(cfg):\n    from pycocoevalcap.meteor.meteor import Meteor\n    from pycocoevalcap.spice.spice import Spice\n    java_path(Path(cfg['test_output']))\n\n    import subprocess, sys\n    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\n    from pycocoevalcap.cider.cider import Cider\n    from pycocoevalcap.bleu.bleu import Bleu\n    from pycocoevalcap.rouge.rouge import Rouge\n    from saved_quality_data import chair_counts\n    out = Path(cfg['output'])\n    lock = checked_lock(out)\n    inputs = read(out / 'inference_inputs.json')\n    beams = read_beams(out, lock)\n    if sha(out / 'inference_inputs.json') != lock['inference_sha256'] or sha(out / 'test_references.json') != lock['references_sha256']:\n        raise ValueError('Changed locked inputs')\n    scored = {}\n    for rank in (0, 1):\n        w = read(out / f'worker{rank}.json')\n        path = out / f'scores_rank{rank}.jsonl'\n        if not w['complete'] or w['images'] != 2500 or w['scored_pairs'] != 12500 or (sha(path) != w['scores_sha256']):\n            raise ValueError('Incomplete TEST scoring worker')\n        rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]\n        if [r['coco_id'] for r in rows] != [r['coco_id'] for r in inputs[rank::2]]:\n            raise ValueError('Wrong TEST score IDs')\n        scored.update({r['coco_id']: r for r in rows})\n    if len(scored) != 5000:\n        raise ValueError('Missing TEST scores')\n    chosen = []\n    for row, beam in zip(inputs, beams):\n        s = scored[row['coco_id']]\n        if s['image_sha256'] != row['image_sha256']:\n            raise ValueError('Wrong image-score identity')\n        k = fixed_rank(beam, s['clipscore'], row['detected'])\n        if k != s['selected_rank']:\n            raise ValueError('Fixed selection replay differs')\n        chosen.append(k)\n    save(out / 'selected_ranks.json', chosen)\n    refs = read(out / 'test_references.json')\n    ptb = PTBTokenizer()\n    ref = ptb.tokenize({r['coco_id']: [dict(caption=c) for c in r['references']] for r in refs})\n    chair = Path(cfg['chair'])\n    metrics = {}\n    flags = {}\n    if sha(chair / 'chair.pkl') != '73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3':\n        raise ValueError('Changed CHAIR cache')\n    for policy in POLICIES:\n        selected = [dict(coco_id=r['coco_id'], caption=r['candidates'][0 if policy == 'decoder_rank0' else k]['caption']) for r, k in zip(beams, chosen)]\n        hyp = ptb.tokenize({r['coco_id']: [dict(caption=r['caption'])] for r in selected})\n        c, per = Cider().compute_score(ref, hyp)\n        b, _ = Bleu(4).compute_score(ref, hyp)\n        r, _ = Rouge().compute_score(ref, hyp)\n        ci = out / (policy + '_chair_input.json')\n        co = out / (policy + '_chair_output.json')\n        save(ci, [dict(image_id=r['coco_id'], caption=r['caption']) for r in selected])\n        subprocess.run([sys.executable, '-u', chair / 'chair.py', '--cap_file', ci, '--cache', chair / 'chair.pkl', '--save_path', co], cwd=chair, check=True, timeout=300)\n        ch = read(co)\n        m, counts = chair_counts(ch, selected)\n        meteor = Meteor()\n        try:\n            meteor_score, meteor_per = meteor.compute_score(ref, hyp)\n        finally:\n            del meteor\n        spice_score, spice_per = Spice().compute_score(ref, hyp)\n        if len(meteor_per) != 5000 or len(spice_per) != 5000 or not np.isfinite([meteor_score, spice_score]).all():\n            raise ValueError('Incomplete METEOR/SPICE')\n        metrics[policy] = dict(CIDEr=float(c), BLEU_1=float(b[0]), BLEU_4=float(b[3]), METEOR=float(meteor_score), SPICE=float(spice_score), ROUGE_L=float(r), **m, counts=counts, mean_words=sum((len(r['caption'].split()) for r in selected)) / 5000, short_le2=sum((len(r['caption'].split()) <= 2 for r in selected)))\n        save(out / (policy + '_captions.json'), selected)\n        save(out / (policy + '_cider_per_image.json'), [dict(coco_id=int(i), CIDEr=float(v)) for i, v in zip(ref, per)])\n        flags[policy] = {r['image_id']: bool(r['mscoco_hallucinated_words']) for r in ch['sentences']}\n    a, b = [flags[p] for p in POLICIES]\n    changes = dict(changed_captions=sum((k != 0 for k in chosen)), rank_histogram={str(k): chosen.count(k) for k in range(5)}, CHAIR_repairs=sum((a[i] and (not b[i]) for i in a)), CHAIR_new_flags=sum((not a[i] and b[i] for i in a)))\n    result = dict(status='CSA_PER_CAPTION_FIXED_RERANK_VALID5000_COMPLETE', images=5000, test_used=False, training=False, checkpoint_sha256=lock['checkpoint_sha256'], metrics=metrics, changes=changes, selector=lock['selector'], tuned_on_test=False, oracle_used_for_selection=False, prior_project_test_exposure=True, selection='both prelocked policies reported; fixed VALID164 policy', NLL_note='same frozen captioner; reranking changes output only, no target-conditioned pass')\n    save(out / 'result.json', result)\n    print(json.dumps(result, indent=2), flush=True)\n\nif __name__ == '__main__':\n    parser = argparse.ArgumentParser()\n    parser.add_argument('mode', choices=('prefetch', 'text_preflight', 'worker', 'score'))\n    parser.add_argument('config')\n    args = parser.parse_args()\n    globals()[args.mode](read(args.config))\n",encoding='utf8')

    rerank_out=OUTPUT/'rerank';rerank_out.mkdir()
    rerank_cfg=dict(output=str(rerank_out),test_output=str(OUTPUT),dataset_json=str(COCO_JSON),
                    detections=str(DETECTIONS),chair=str(CHAIR),model_cache=str(SOURCE/'hf_cache'),phase_seconds=2100)
    rerank_config=SOURCE/'rerank_config.json'
    rerank_config.write_text(json.dumps(rerank_cfg))
    run([sys.executable,'-u',SOURCE/'csa_per_caption_rerank_data.py',rerank_config],900)
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,
         'transformers==4.57.1','tokenizers==0.22.1','huggingface-hub==0.36.2','safetensors==0.6.2'],300)
    script=SOURCE/'run_csa_per_caption_rerank.py'
    run([sys.executable,'-u',script,'prefetch',rerank_config],1200)
    run([sys.executable,'-u',script,'text_preflight',rerank_config],180)
    run([*launch,script,'worker',rerank_config],2130)
    run([sys.executable,'-u',script,'score',rerank_config],3600)
    result=json.loads((rerank_out/'result.json').read_text())
    complete=result['status']=='CSA_PER_CAPTION_FIXED_RERANK_VALID5000_COMPLETE'
    if not complete:raise ValueError('Incomplete rerank scoring')
except BaseException:
    (OUTPUT/'failure.txt').write_text(redact(traceback.format_exc()));raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,training=False,test_tuning=False,source_version='csa_structure_per_caption_full_xe10_resume124177')))
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUTPUT.rglob('*'):
            if p.is_file() and p.suffix in ('.json','.jsonl','.log','.txt'):z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',review,flush=True);print('KEEP OUTPUT:',OUTPUT,flush=True)
    print('TEST COMPLETE; no training/tuning.' if complete else 'INCOMPLETE; preserve captions/logs; no automatic re-decode or retraining.',flush=True)
